# 4.4 因子分解机：FM

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

单独掌握稀疏二阶交互：从 MovieLens 评分阈值派生的高偏好标签、FM 数学恒等式，到 PyTorch 训练、概率推理、AUC 与 LogLoss；该教学任务不冒充曝光—点击日志。

## Setup

本 Notebook 的默认真实数据是 **MovieLens latest-small（smoke 确定性切片）；full 用官方完整 MovieLens latest（约 33M 评分）**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Factorization Machines](https://www.csie.ntu.edu.tw/~b97053/paper/Rendle2010FM.pdf)

In [1]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
DATASET_KEY = "movielens"
# Setup 只声明执行边界。完整数据由章节 runner 在 Train & Inference 单元按需读取，
# 避免仅打开 Notebook 就解析数千万行文件。
REAL_DATASET = {
    "dataset": DATASET_KEY,
    "profile": PROFILE,
    "loading": "lazy: chapter runner owns loading and returns executed provenance",
    "randomly_fabricated_rows": 0,
}
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "dataset_boundary": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

{'profile': 'smoke', 'project_root': '<ARTIFACT_ROOT>', 'artifact_root': '<ARTIFACT_ROOT>', 'dataset_boundary': {'dataset': 'movielens', 'profile': 'smoke', 'loading': 'lazy: chapter runner owns loading and returns executed provenance', 'randomly_fabricated_rows': 0}, 'cuda_available': True, 'cuda_device': 'NVIDIA RTX 6000 Ada Generation'}


## 来源论文与阅读提示

**Rendle (2010), Factorization Machines** 的核心贡献是把矩阵分解的隐向量交互推广到任意稀疏特征，并通过代数恒等式在线性时间内计算全部二阶项。阅读时应特别看稀疏条件下“未直接观察过的特征组合”如何借各自隐向量共享统计。

### 公式怎么读（直觉版）

先把 one-hot 理解成一排开关：当前用户、物品、设备对应的开关为 1，其余为 0。FM 为每个开关配置一张小坐标卡；任意两个打开的开关都做一次点积。恒等式不是新模型，只是把“逐对计算”重新整理为“先求和再平方”，避免重复工作。相关向量、点积和概率知识见 **3.0**。

In [2]:
# Bounded teaching view：无论正式 PROFILE 是 smoke 还是 full，这一段公式演示都只读取
# 仓库内的 MovieLens latest-small 确定性小视图。正式实验由后面的 chapter runner 单独加载。
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, roc_auc_score, log_loss
from recsys_lab import data as data_tools
from recsys_lab.data import leave_last_out, movielens_provenance

SEED = 2026
torch.manual_seed(SEED)
_formal_profile = os.environ["RECSYS_PROFILE"]
try:
    os.environ["RECSYS_PROFILE"] = "smoke"
    data_tools._load_cached.cache_clear()
    ratings, movies = data_tools.load_movielens(max_users=48, max_items=360, min_user_events=12)
finally:
    os.environ["RECSYS_PROFILE"] = _formal_profile
    data_tools._load_cached.cache_clear()

train_ratings, test_ratings = leave_last_out(ratings)
n_users, n_items = ratings.user_id.nunique(), ratings.item_id.nunique()
TEACHING_DATASET = movielens_provenance(ratings)
assert TEACHING_DATASET["profile"] == "smoke"
assert TEACHING_DATASET["randomly_fabricated_rows"] == 0
print({
    "mode": "bounded teaching view (not the formal experiment)",
    "rows": len(ratings), "users": n_users, "items": n_items,
    "train_rows": len(train_ratings), "test_rows": len(test_ratings),
    "source": TEACHING_DATASET["source"], "fabricated_rows": 0,
})
display(ratings[["userId", "movieId", "rating", "timestamp", "title", "genres"]].head(8))

{'mode': 'bounded teaching view (not the formal experiment)', 'rows': 10487, 'users': 48, 'items': 360, 'train_rows': 10439, 'test_rows': 48, 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'fabricated_rows': 0}


,userId,movieId,rating,timestamp,title,genres
0,140,318,4.0,942840891,"Shawshank Redemption, The (1994)",Crime|Drama
1,140,527,5.0,942840891,Schindler's List (1993),Drama|War
2,140,1221,3.0,942840891,"Godfather: Part II, The (1974)",Crime|Drama
3,140,50,3.0,942840991,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,140,595,5.0,942840991,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX
5,140,1198,4.0,942841170,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
6,140,2028,5.0,942841170,Saving Private Ryan (1998),Action|Drama|War
7,140,1721,4.0,942841219,Titanic (1997),Drama|Romance


---

## 4. 因子分解机（FM）：为稀疏特征学习二阶交互

### 4.1 为什么 MF 不够？

**通用先修：** [3.2 样本、特征与标签](/notebooks/3_2_data_ml_basics#observation-label) · [3.3 逐元素与点积](/notebooks/3_3_linear_algebra#elementwise-dot) · [3.6 二元交叉熵](/notebooks/3_6_information_theory#cross-entropy-kl)<br>
**本论文新增数学：** 用共享隐向量参数化任意稀疏特征的二阶交互，并把 $O(n^2k)$ 的逐对计算化成 $O(nk)$ 恒等式。

CTR 排序不只有 `user_id` 和 `item_id`，还会有设备、小时、地域、品类、价格等上下文。直接为每一对特征做 one-hot 交叉会导致参数爆炸；普通 LR 又只能做加法，无法表达“某用户群在晚间更喜欢某品类”。

FM 的输入是一个超长稀疏向量 $x\in\mathbb R^n$：每个特征占一格，one-hot 类别特征取 0/1，连续特征取实数值。模型为每个特征 $i$ 学一个 $k$ 维隐向量 $v_i\in\mathbb R^k$，用内积 $\langle v_i,v_j\rangle=\sum_{f=1}^{k}v_{if}v_{jf}$ 共享所有二阶交叉统计：

$$
\hat y(x)=w_0+\sum_i w_ix_i+\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
$$

三项分别是：$w_0$ 全局偏置；$\sum_i w_ix_i$ 一阶线性项（与 LR 相同）；$\sum_{i<j}\langle v_i,v_j\rangle x_ix_j$ 二阶交叉项——只有同时“打开”（$x_i,x_j$ 都非零）的特征对才参与。关键在于稀疏性：即使某对特征从未在训练集共同出现，它们各自与其他特征的共现也已把 $v_i,v_j$ 学好，交叉强度仍能由内积给出。

朴素计算二阶项要枚举全部 $\binom{n}{2}$ 对特征、每对做一次 $k$ 维内积，复杂度 $O(n^2k)$。利用“和的平方减去平方的和”恒等式可化为 $O(nk)$：

$$
\sum_{i<j}\langle v_i,v_j\rangle x_ix_j
=\frac12\sum_f\left[\left(\sum_i v_{if}x_i\right)^2-\sum_i v_{if}^2x_i^2\right]
$$

推导只需一步代数：把 $(\sum_i v_{if}x_i)^2$ 展开会得到所有有序对 $(i,j)$ 的乘积——包含 $i=j$ 的对角项，且每对 $i<j$ 出现两次；减去对角项 $\sum_i v_{if}^2x_i^2$ 再乘 $\tfrac12$ 去重，正好回到左边。右侧只需对每个因子维度 $f$ 各扫一遍特征。

**手算一遍。** 取 $k=2$、两个特征 $v_1=[1,2]$、$v_2=[0.5,-1]$，且 $x_1=x_2=1$。左边：$\langle v_1,v_2\rangle=1\times0.5+2\times(-1)=-1.5$。右边逐维算：$f=1$ 时 $(1+0.5)^2-(1^2+0.5^2)=2.25-1.25=1$；$f=2$ 时 $(2+(-1))^2-(2^2+(-1)^2)=1-5=-4$；合起来 $\tfrac12[1+(-4)]=-1.5$，与左边一致。

当特征只有 user one-hot 和 item one-hot 时，FM 的交叉项就退化为 MF；因此 FM 可以理解为 MF 对通用稀疏特征的扩展。

### 4.2 数据：评分阈值派生“高偏好”标签（不是曝光—点击日志）

MovieLens 记录的是用户主动提交的星级，**没有完整曝光分母**。这里把真实评分 `rating >= 4.0` 派生为高偏好标签、较低评分派生为 0，只用于演示 FM 的二分类、AUC 与 LogLoss 计算；变量名 `click` 是代码兼容别名，不代表真实广告点击。没有评分的 user–item 仍是未知，不能补成曝光未点击。真实 CTR 结论必须改用包含曝光与点击的日志。

In [3]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ctr = ratings.sort_values("timestamp").tail(5000).copy().reset_index(drop=True)
# `click` 是任务别名；标签由真实评分 rating >= 4.0 确定，不使用随机采样。
ctr["click"] = ctr["like"].astype(int)
split = int(len(ctr) * .8)  # 严格按真实 timestamp 切分
categorical = ["user_id", "item_id", "genre_id", "hour", "decade_id"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_sparse = encoder.fit_transform(ctr.loc[:split-1, categorical])
test_sparse = encoder.transform(ctr.loc[split:, categorical])
scaler = StandardScaler()
train_numeric = scaler.fit_transform(ctr.loc[:split-1, ["item_popularity", "user_activity"]])
test_numeric = scaler.transform(ctr.loc[split:, ["item_popularity", "user_activity"]])
fm_train_x = np.c_[train_sparse, train_numeric].astype("float32")
fm_test_x = np.c_[test_sparse, test_numeric].astype("float32")
ctr_train_y = ctr.loc[:split-1, "click"].to_numpy()
ctr_test_y = ctr.loc[split:, "click"].to_numpy()
print({"train": fm_train_x.shape, "test": fm_test_x.shape, "train_positive_rate": round(ctr_train_y.mean(), 3)})

{'train': (4000, 437), 'test': (1000, 437), 'train_positive_rate': np.float64(0.474)}


### 4.3 训练：显式实现 FM 的线性时间交叉

In [4]:
class FactorizationMachine(torch.nn.Module):
    def __init__(self, n_features, factor_dim=10):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)
        self.factors = torch.nn.Parameter(torch.randn(n_features, factor_dim) * 0.03)

    def forward(self, x):
        linear_logit = self.linear(x).squeeze(1)
        summed = x @ self.factors
        squared_sum = summed.pow(2)
        sum_squared = x.pow(2) @ self.factors.pow(2)
        pairwise_logit = 0.5 * (squared_sum - sum_squared).sum(dim=1)
        return linear_logit + pairwise_logit

fm_model = FactorizationMachine(fm_train_x.shape[1], factor_dim=10)
fm_optimizer = torch.optim.Adam(fm_model.parameters(), lr=.025, weight_decay=1e-5)
x_train_tensor = torch.tensor(fm_train_x)
y_train_tensor = torch.tensor(ctr_train_y, dtype=torch.float32)

for epoch in range(100):
    fm_logit = fm_model(x_train_tensor)
    fm_loss = torch.nn.functional.binary_cross_entropy_with_logits(fm_logit, y_train_tensor)
    fm_optimizer.zero_grad(); fm_loss.backward(); fm_optimizer.step()

print({"FM final BCE": round(float(fm_loss.detach()), 4)})

{'FM final BCE': 0.0102}


### 4.4 推理与测试：输出概率、AUC 与 LogLoss

In [5]:
fm_model.eval()
with torch.no_grad():
    fm_probability = torch.sigmoid(fm_model(torch.tensor(fm_test_x))).numpy()
fm_auc = float(roc_auc_score(ctr_test_y, fm_probability))
fm_logloss = float(log_loss(ctr_test_y, fm_probability))
display(pd.DataFrame({"label": ctr_test_y[:8], "p_click": fm_probability[:8].round(4)}))
print({"FM AUC": round(fm_auc, 4), "FM LogLoss": round(fm_logloss, 4)})

,label,p_click
0,0,0.0028
1,1,0.2287
2,0,0.0021
3,0,0.1610
4,0,0.0002
5,0,0.0159
6,0,0.3017
7,0,0.0062


{'FM AUC': 0.5964, 'FM LogLoss': 2.6061}


### 4.5 结果讨论与边界

AUC 关注正样本是否排在负样本前，LogLoss 关注概率本身是否可信。FM 能通过隐向量共享低频交叉的统计，但所有交互仍是二阶且形式相同。

**优点**：适合超稀疏 one-hot；无需枚举交叉；参数能跨组合共享；计算复杂度线性。  
**缺点**：主要建模二阶；不理解行为顺序；所有 field 共用同一种内积。  
**常见升级**：FFM 为不同 field 学不同向量；DeepFM 共享 embedding，同时加入 DNN 学高阶非线性。

## Train & Inference（正式实验）

下方唯一的正式指标入口调用 `chapter_code/<slug>/train.py`。runner 会按 `PROFILE`
惰性加载对应完整/CI 数据、执行内存受控训练与评测，并持续打印阶段进度。上面的
bounded teaching view 只用于手算和理解代码，不会写入章节结果。

In [6]:
from importlib import import_module
from recsys_lab.runtime import print_progress

chapter_train = import_module("chapter_code.4_4_factorization_machine.train")
result = chapter_train.train_and_evaluate(
    epochs=8 if PROFILE == "smoke" else 8,
    progress=print_progress,
)
REAL_DATASET = result["dataset"]
_executed_fabricated_rows = (
    REAL_DATASET.get("randomly_fabricated_rows", result.get("randomly_fabricated_rows"))
    if isinstance(REAL_DATASET, dict)
    else result.get("randomly_fabricated_rows")
)
assert _executed_fabricated_rows == 0
print({"profile": PROFILE, "executed_dataset": REAL_DATASET})
print({key: value for key, value in result.items() if key not in {"dataset", "loss_curve"}})

[data_prepare] 0/1 加载并编码 MovieLens 特征


[data_prepare] 1/1 rows=4200


[train] 0/8 训练 Factorization Machine


[train] 1/8 loss=4.4897


[train] 2/8 loss=4.1403


...

[inference] 0/1 分批生成测试集概率


[inference] 1/1


[evaluate] 0/1 计算 AUC 与 LogLoss


[evaluate] 1/1 auc=0.540819 logloss=3.41032


{'profile': 'smoke', 'executed_dataset': 'MovieLens latest-small'}
{'randomly_fabricated_rows': 0, 'auc': 0.5408193094277329, 'logloss': 3.4103241581248875}


## Checks

bounded teaching view 确认概率合法、AUC 高于随机水平并观察 LogLoss；正式产物来自 runner。

In [7]:
assert np.all((fm_probability >= 0) & (fm_probability <= 1))
assert fm_auc > .55
assert np.isfinite(fm_logloss)
print('PASS：FM 训练、概率推理和 CTR 指标均有效。')

PASS：FM 训练、概率推理和 CTR 指标均有效。


In [8]:
result_dir = ARTIFACT_ROOT / "results" / "chapter_4"; result_dir.mkdir(parents=True, exist_ok=True)
payload = {"records": [{"algorithm":"FM", "task":"CTR", "metric":"AUC ↑", "value":result["auc"],
                       "secondary_metric":"LogLoss ↓", "secondary_value":result.get("logloss"),
                       "status":"executed", "online_inference":"线性项 + 二阶隐向量交互",
                       "source_notebook":"4_4_factorization_machine"}]}
(result_dir / "factorization_machine.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写出章节指标：factorization_machine.json")

已写出章节指标：factorization_machine.json


## Next Steps

加入 LR 基线；拆解不同 field 的交互；继续学习 FFM、DeepFM 与 DCN。